In [39]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from unidecode import unidecode
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler


In [30]:
def build_stats(filename_full='stats_full', filename_reduced='stats_reduced'):
    df = pd.read_csv('unified_player_stats.csv')
    df = df.loc[df['season'] == "2025-26"]
    df = df[['player', '_name_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots_per90', 'shots_on_target', 'shots_inside_box', 'shots_outside_box', 'npxg_per90', 'npxg_overperformance', 'goal_conversion_pct', 'big_chances_missed', 'key_passes_per90', 'xag_per90', 'attempt_assists', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'pass_to_assist', 'passes_total', 'pass_completion_pct', 'passes_final_third', 'passes_opp_half', 'long_balls_total', 'long_balls_pct', 'crosses_total', 'crosses_pct', 'chipped_passes_total', 'chipped_passes_pct', 'touches', 'dribbles_per90', 'dribbles_pct', 'dispossessed', 'possession_lost', 'ball_recoveries', 'possession_won_att_third', 'tackles', 'tackles_won_pct', 'interceptions_per90', 'clearances', 'blocked_shots', 'dribbled_past', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls', 'fouled']]
    df['shots_on_target_per90'] = df['shots_on_target']/df['ninety_s']
    df['shots_inside_box_per90'] = df['shots_inside_box']/df['ninety_s']
    df['shots_outside_box_per90'] = df['shots_outside_box']/df['ninety_s']
    df['big_chances_missed_per90'] = df['big_chances_missed']/df['ninety_s']
    df['attempt_assists_per90'] = df['attempt_assists']/df['ninety_s']
    df['pass_to_assist_per90'] = df['pass_to_assist']/df['ninety_s']
    df['passes_per90'] = df['passes_total']/df['ninety_s']
    df['passes_final_third_per90'] = df['passes_final_third']/df['ninety_s']
    df['passes_opp_half_per90'] = df['passes_opp_half']/df['ninety_s']
    df['long_balls_per90'] = df['long_balls_total']/df['ninety_s']
    df['crosses_per90'] = df['crosses_total']/df['ninety_s']
    df['chipped_passes_per90'] = df['chipped_passes_total']/df['ninety_s']
    df['touches_per90'] = df['touches']/df['ninety_s']
    df['dispossessed_per90'] = df['dispossessed']/df['ninety_s']
    df['possession_lost_per90'] = df['possession_lost']/df['ninety_s']
    df['ball_recoveries_per90'] = df['ball_recoveries']/df['ninety_s']
    df['possession_won_att_third_per90'] = df['possession_won_att_third']/df['ninety_s']
    df['tackles_per90'] = df['tackles']/df['ninety_s']
    df['clearances_per90'] = df['clearances']/df['ninety_s']
    df['blocked_shots_per90'] = df['blocked_shots']/df['ninety_s']
    df['dribbled_past_per90'] = df['dribbled_past']/df['ninety_s']
    df['fouls_per90'] = df['fouls']/df['ninety_s']
    df['fouled_per90'] = df['fouled']/df['ninety_s']
    df['npxg_overperformance_per90'] = df['npxg_overperformance']/df['ninety_s']
    # df['player_norm'] = df['_name_norm']
    df['player_norm'] = df['player'].apply(unidecode)

    df.to_csv(f"{filename_full}.csv", index=False, header=True)
    df_reduced = df[['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots_per90', 'shots_on_target_per90', 'shots_inside_box_per90', 'shots_outside_box_per90', 'npxg_per90', 'npxg_overperformance_per90', 'goal_conversion_pct', 'big_chances_missed_per90', 'key_passes_per90', 'xag_per90', 'attempt_assists_per90', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'pass_to_assist_per90', 'passes_per90', 'pass_completion_pct', 'passes_final_third_per90', 'passes_opp_half_per90', 'long_balls_per90', 'long_balls_pct', 'crosses_per90', 'crosses_pct', 'chipped_passes_per90', 'chipped_passes_pct', 'touches_per90', 'dribbles_per90', 'dribbles_pct', 'dispossessed_per90', 'possession_lost_per90', 'ball_recoveries_per90', 'possession_won_att_third_per90', 'tackles_per90', 'tackles_won_pct', 'interceptions_per90', 'clearances_per90', 'blocked_shots_per90', 'dribbled_past_per90', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls_per90', 'fouled_per90']]
    df_reduced.to_csv(f"{filename_reduced}.csv", index=False, header=True)

In [31]:
def build_df(filename='stats_reduced', minutes=1500):

    df = pd.read_csv('stats_reduced.csv')
    df_min = df.loc[df['minutes'] >= minutes]
    df_min = df_min.loc[(df['pos'] != 'GK') & (df['pos'] != 'GK S')]
    df_min = df_min.dropna(subset=['pos'])
    df_min = df_min.reset_index()
    df_min = df_min.drop(columns=['index'])
    df_min['goal_conversion_pct'] = df_min['goal_conversion_pct']/100
    df_min['pass_completion_pct'] = df_min['pass_completion_pct']/100
    df_min['long_balls_pct'] = df_min['long_balls_pct']/100
    df_min['crosses_pct'] = df_min['crosses_pct']/100
    df_min['chipped_passes_pct'] = df_min['chipped_passes_pct']/100
    df_min['dribbles_pct'] = df_min['dribbles_pct']/100
    df_min['tackles_won_pct'] = df_min['tackles_won_pct']/100
    df_min['aerials_won_pct'] = df_min['aerials_won_pct']/100
    df_min['ground_duels_won_pct'] = df_min['ground_duels_won_pct']/100
    df_min['duels_won_pct'] = df_min['duels_won_pct']/100


    return df_min

In [32]:
def standardize_df(df):
    scaler = preprocessing.StandardScaler()
    data_df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    standard_df = scaler.fit_transform(data_df)
    standard_df = pd.DataFrame(standard_df, columns=data_df.columns)
    standard_df.insert(0, 'player', df['player'])
    standard_df.insert(0, 'player_norm', df['player_norm'])
    standard_df.insert(2, 'team', df['team'])
    standard_df.insert(3, 'pos', df['pos'])
    standard_df.insert(4, 'minutes', df['minutes'])
    standard_df.insert(5, 'ninety_s', df['ninety_s'])
    standard_df.insert(6, 'games', df['games'])
    return standard_df

In [58]:
def apply_pca(df_standard):
    df_pca = df_standard.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    pca = PCA(n_components=7)
    data_pca = pca.fit_transform(df_pca)
    data_pca = pd.DataFrame(data_pca)
    data_pca.insert(0, 'player', df_standard['player'])
    data_pca.insert(1, 'player_norm', df_standard['player_norm'])
    data_pca.insert(2, 'team', df_standard['team'])
    data_pca.insert(3, 'pos', df_standard['pos'])
    data_pca.insert(4, 'minutes', df_standard['minutes'])
    data_pca.insert(5, 'ninety_s', df_standard['ninety_s'])
    data_pca.insert(6, 'games', df_standard['games'])

    cont_df = pd.DataFrame()
    feature_names = df_pca.columns
    cont0 = []
    cont1 = []
    cont2 = []
    cont3 = []
    cont4 = []
    cont5 = []
    cont6 = []
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    for i in range(len(feature_names)):
        # print(f"Feature: {feature_names[i]}, Loadings: {loadings[i]}")
        cont0.append(loadings[i][0])
        cont1.append(loadings[i][1])
        cont2.append(loadings[i][2])
        cont3.append(loadings[i][3])
        cont4.append(loadings[i][4])
        cont5.append(loadings[i][5])
        cont6.append(loadings[i][6])
    cont_df['feature'] = feature_names
    cont_df['cont_0'] = cont0
    cont_df['cont_1'] = cont1
    cont_df['cont_2'] = cont2
    cont_df['cont_3'] = cont3
    cont_df['cont_4'] = cont4
    cont_df['cont_5'] = cont5
    cont_df['cont_6'] = cont6
    # print(f"PCA Explained Variance: {pca.explained_variance_ratio_}")


    loading_matrix = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[
            'PC1', 'PC2', 'PC3',
            'PC4', 'PC5', 'PC6', 'PC7'
        ]
    )

    loading_matrix['loading_strength'] = np.sqrt(
        (loading_matrix ** 2).sum(axis=1)
    )

    loading_matrix.sort_values(
        'loading_strength',
        ascending=False
    )
    return data_pca, cont_df, loading_matrix

In [55]:
def get_distances(player_name, df_pca):
    search_row = df_pca.loc[df_pca['player_norm'] == player_name]
    dist_df = pd.DataFrame()
    players = []
    distances = []
    cos_sims = []
    teams = []
    positions = []

    

    for i in tqdm(df_pca.index, "Computing distances"):
        compare_row = df_pca.loc[i]
        search_player = search_row['player_norm']
        compare_player = compare_row['player_norm']
        compare_team = compare_row['team']
        compare_pos = compare_row['pos']
        search_data = search_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        compare_data = compare_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        distance = np.linalg.norm(search_data - compare_data)
        cos_sim = cosine_similarity(search_data.reshape(1, -1), compare_data.reshape(1, -1))[0][0]
        # print(f"Distance between {search_player} and {compare_player}: {distance}")
        players.append(compare_player)
        distances.append(distance)
        cos_sims.append(cos_sim)
        teams.append(compare_team)
        positions.append(compare_pos)

    
    
    dist_df['player'] = players
    dist_df['team'] = teams
    dist_df['position'] = positions
    dist_df['distance'] = distances
    dist_df['cosine_similarity'] = cos_sims
    dist_df = dist_df.sort_values(by=['distance'], ascending=True)
    dist_df = dist_df.reset_index(drop=True)
    dist_df['rank_distance'] = dist_df['distance'].rank(method='min')
    dist_df['rank_cosine_similarity'] = dist_df['cosine_similarity'].rank(ascending=False, method='min')
    dist_df['rank_distance'] = dist_df['rank_distance'].astype(int)
    dist_df['rank_cosine_similarity'] = dist_df['rank_cosine_similarity'].astype(int)

    return dist_df

In [66]:
build_stats()
df = build_df(minutes=1000)
df_standard = standardize_df(df)
df_pca, cont_df, loadings = apply_pca(df_standard)
# df_pca.head()
print(loadings['loading_strength'].sort_values(ascending=False))

C:\Users\an.capobianco\AppData\Local\Temp\ipykernel_12796\339070016.py:2: DtypeWarning: Columns (0: pos) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('unified_player_stats.csv')


npxg_overperformance_per90        0.769452
goal_conversion_pct               0.655311
ground_duels_won_pct              0.536942
chipped_passes_pct                0.508729
duels_won_pct                     0.506667
long_balls_pct                    0.501088
xg_buildup_per90                  0.477712
fouls_per90                       0.450703
xg_chain_per90                    0.444538
fouled_per90                      0.442460
crosses_per90                     0.425200
pass_completion_pct               0.421603
big_chances_missed_per90          0.416098
clearances_per90                  0.415583
shots_inside_box_per90            0.407110
dribbles_pct                      0.401826
key_passes_per90                  0.399229
dribbled_past_per90               0.396437
npxg_per90                        0.396257
xag_per90                         0.393455
dribbles_per90                    0.388102
long_balls_per90                  0.384820
passes_per90                      0.377070
shots_on_ta

In [ ]:
print(cont_df[['feature', 'cont_0']].sort_values(key=abs,by=['cont_0'], ascending=False).head(10))
print(cont_df[['feature', 'cont_1']].sort_values(key=abs,by=['cont_1'], ascending=False).head(10))
print(cont_df[['feature', 'cont_2']].sort_values(key=abs,by=['cont_2'], ascending=False).head(10))
print(cont_df[['feature', 'cont_3']].sort_values(key=abs,by=['cont_3'], ascending=False).head(10))
print(cont_df[['feature', 'cont_4']].sort_values(key=abs,by=['cont_4'], ascending=False).head(10))
print(cont_df[['feature', 'cont_5']].sort_values(key=abs,by=['cont_5'], ascending=False).head(10))
print(cont_df[['feature', 'cont_6']].sort_values(key=abs,by=['cont_6'], ascending=False).head(10))

In [54]:
player_name = "Federico Dimarco"
sim_df = get_distances(player_name, df_pca)
sim_df

RangeIndex(start=0, stop=1423, step=1)


Computing distances: 100%|██████████| 1423/1423 [00:00<00:00, 1435.41it/s]


,player,team,position,distance,cosine_similarity,rank_distance,rank_cosine_similarity
0,Federico Dimarco,Inter,D S,0.000000,1.000000,1,1
1,Bruno Fernandes,Manchester United,M,2.398073,0.956028,2,3
2,Alex Grimaldo,Bayer Leverkusen,D,2.785005,0.969816,3,2
3,Florian Thauvin,Lens,M S,2.863920,0.931641,4,5
4,Mathis Cherki,Manchester City,F M S,3.017899,0.947683,5,4
...,...,...,...,...,...,...,...
1418,Ademola Lookman,"Atalanta,Atletico Madrid",F M S,17.394791,0.586667,1419,164
1419,Angel Gomes,"Marseille,Wolverhampton Wanderers",M S,17.672619,0.373364,1420,319
1420,Xavi Simons,"RasenBallsport Leipzig,Tottenham",F M S,18.456276,0.420164,1421,288
1421,Diego Coppola,"Brighton,Paris FC",D,21.926462,-0.403942,1422,1001


In [12]:
df = pd.read_csv('stats_reduced.csv')
df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s'])
columns = df.columns
corr = df[columns].corr()
corr_pairs = (
    corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(corr_pairs.head(30))

shots_on_target_per90           shots_inside_box_per90            0.987413
passes_per90                    touches_per90                     0.985855
touches_per90                   ball_recoveries_per90             0.982383
passes_final_third_per90        passes_opp_half_per90             0.981506
passes_per90                    passes_opp_half_per90             0.979599
shots_inside_box_per90          dispossessed_per90                0.975089
shots_on_target_per90           dispossessed_per90                0.973291
passes_final_third_per90        ball_recoveries_per90             0.972539
passes_per90                    ball_recoveries_per90             0.970899
passes_opp_half_per90           touches_per90                     0.969315
passes_final_third_per90        touches_per90                     0.967557
possession_won_att_third_per90  blocked_shots_per90               0.965875
shots_on_target_per90           big_chances_missed_per90          0.963151
xg_chain_per90           